# Disease grouping after build graph

Pipeline: **rule-based grouping → BERT clustering → expert review (Hoai + Claude) → apply `MONDO_grouped`**.

| Step | Input | Output |
|------|---------|--------|
| 1 | `{RUN_DATE}-kg_giant.csv` | `disease_nodes`, `disease_group_1` |
| 2 | BERT embeddings | `cos_sim`, `review_df` |
| 3 | Dup-name fixes CSV + Claude round1/2 | `review_final` |
| 4 | Expert map | `kg_grouped_diseases.csv` |
| 5 | Group map | `{RUN_DATE}-primekg_plus.csv` |

**Claude decisions (pre-computed, no re-query needed):**
- `20260529_claude_round1_decisions.csv` — mergeable (≠ `no`) *(formerly: `..._claude_updated.csv`)*
- `20260529_claude_round2_no_with_cui.csv` — confirmed `no` *(formerly: `..._help2_expaned_with_cui.csv`)*

Run top-to-bottom. Flags: `RECOMPUTE_EMBEDDINGS`, `RECOMPUTE_BERT_REVIEW` (default `False` = use saved files).

Configure paths in the **first code cell** (`resolve_primekg_root()`). Override with `export PRIMEKG_ROOT=/path/to/PrimeKG` if needed.


In [1]:
from pathlib import Path
from tqdm.notebook import tqdm
import re
import os
import shutil
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
import torch
from transformers import AutoTokenizer, AutoModel

%load_ext autoreload
%autoreload 2


def resolve_primekg_root() -> Path:
    """Locate PrimeKG repo root (contains datasets/data)."""
    if env := os.environ.get("PRIMEKG_ROOT"):
        root = Path(env).expanduser().resolve()
        if (root / "datasets" / "data" / "ppi").is_dir():
            return root
        raise FileNotFoundError(f"PRIMEKG_ROOT invalid (no datasets/data): {root}")

    for start in (Path.cwd().resolve(),):
        for parent in (start, *start.parents):
            if (parent / "datasets" / "data" / "ppi").is_dir():
                return parent
            if parent.name == "PrimeKG-Plus_release" and (parent.parent / "datasets" / "data" / "ppi").is_dir():
                return parent.parent

    raise FileNotFoundError(
        "Cannot find PrimeKG source data. Export PRIMEKG_ROOT to the repository root "
        "(the folder that contains datasets/data/)."
    )


# ── Paths ──────────────────────────────────────────────────────────────────────
PROJECT_ROOT = resolve_primekg_root()
RELEASE_ROOT = PROJECT_ROOT / "PrimeKG-Plus_release"
KG_SCRIPTS_DIR = PROJECT_ROOT / "knowledge_graph"
RUN_DATE = "20260529"
data_path = PROJECT_ROOT / "datasets" / "data"
save_path = data_path / "kg"
aux_path = save_path / "auxillary"

PRIMEKG_PATH = PROJECT_ROOT / "dataverse_files" / "no_dup_kg.csv"
HOAI_REVIEW_CSV = aux_path / "20260512-kg_grouping_review_candidates_Nurset_done_Hoai checked 260524.csv"
CLAUDE_ROUND1_CSV = KG_SCRIPTS_DIR / f"{RUN_DATE}_claude_round1_decisions.csv"
CLAUDE_ROUND2_CSV = KG_SCRIPTS_DIR / f"{RUN_DATE}_claude_round2_no_with_cui.csv"
KG_GIANT_CSV = aux_path / f"{RUN_DATE}-kg_giant.csv"
EMBED_PATH = aux_path / f"{RUN_DATE}-kg_disease_bert_embeds.npy"
REVIEW_PATH = aux_path / f"{RUN_DATE}-kg_grouping_review_candidates.csv"
DUP_NAME_FIXES_CSV = RELEASE_ROOT / "dataset/auxillary/20260616_dup_name_group_fixes.csv"
if not DUP_NAME_FIXES_CSV.is_file():
    DUP_NAME_FIXES_CSV = KG_SCRIPTS_DIR / "20260616_dup_name_group_fixes.csv"
USE_HOAI_REVIEW = False  # False → use dup-name fixes CSV instead of Hoai review

RECOMPUTE_EMBEDDINGS = False
RECOMPUTE_BERT_REVIEW = False  # False → load saved review CSV; skip BERT clustering
BERT_CUTOFF = 0.98
NAME_DECIDED_LATER = "name decided later"

for pth, label in [
    (KG_GIANT_CSV, "kg_giant"),
    (CLAUDE_ROUND1_CSV, "Claude round1"),
    (CLAUDE_ROUND2_CSV, "Claude round2"),
    (PRIMEKG_PATH, "legacy PrimeKG"),
    (DUP_NAME_FIXES_CSV, "dup name fixes"),
]:
    assert pth.is_file(), f"Missing {label}: {pth}"
if USE_HOAI_REVIEW:
    assert HOAI_REVIEW_CSV.is_file(), f"Missing Hoai review: {HOAI_REVIEW_CSV}"



## 1. Load disease nodes

In [2]:
kg = pd.read_csv(KG_GIANT_CSV, low_memory=False)

disease_nodes = pd.concat([
    kg[["x_id", "x_type", "x_name", "x_source"]].rename(
        columns={"x_id": "node_id", "x_type": "node_type", "x_name": "node_name", "x_source": "node_source"}
    ),
    kg[["y_id", "y_type", "y_name", "y_source"]].rename(
        columns={"y_id": "node_id", "y_type": "node_type", "y_name": "node_name", "y_source": "node_source"}
    ),
])
disease_nodes = (
    disease_nodes.query('node_type == "disease"')
    .drop_duplicates()
    .reset_index(drop=True)
    .reset_index()
    .rename(columns={"index": "node_idx"})
)

## 2. Rule-based grouping

In [3]:
#rule-based string matching
# '''
groups = []
seen = set()
idx2group = {}
no = set()

def isroman(s):
    return bool(re.search(r"^M{0,3}(CM|CD|D?C{0,3})(XC|XL|L?X{0,3})(IX|IV|V?I{0,3})$",s))

def has_subtype_suffix(word):
    """Last token looks like type 1/2, IV, A, etc."""
    return len(word) <= 2 or word.isnumeric() or isroman(word)
    

# Check whether j matches anchor
def rule_names_match(j_name, main_text):
    """Match j to anchor main_text: stripped prefix, exact name, or prefix."""
    j_lower = j_name.strip().lower()
    mt_lower = main_text.strip().lower()
    
    # Case 1: exact match
    #"gaucher disease" == "gaucher disease"  → True
    if j_lower == mt_lower:
        return True

    # Case 2: j starts with anchor
    # "gaucher disease type 1".startswith("gaucher disease ")  → True
    # "gaucher disease, type 1".startswith("gaucher disease,")  → True
    if j_lower.startswith(mt_lower + " ") or j_lower.startswith(mt_lower + ","):
        return True

    # Case 3: j also has subtype suffix → strip suffix then compare
    # "Gaucher disease type 2" → strip "2" → "Gaucher disease" == anchor → True
    j_split = j_name.split(" ")
    j_end = j_split[-1]
    if has_subtype_suffix(j_end):
        m = " ".join(j_split[:-1])
        if m.lower() == mt_lower or same_words(m, main_text):
            return True
    else:
        if same_words(j_name, main_text):
            return True
    return False


def same_words(s1, s2): 
    for word in s1.lower().split(' '): 
        word = word.split(',')[0]
        if word!='type' and word!='(disease)' and word not in s2.lower(): 
            return False 
    for word in s2.lower().split(' '): 
        word = word.split(',')[0]
        if word!='type' and word!='(disease)' and word not in s1.lower(): 
            return False
    return True
#test
# same_words("Fabry disease", "Fabry (disease)")
# same_words("Gaucher disease, type 1", "Gaucher disease")

for i in range(disease_nodes.shape[0]):
    i_name = disease_nodes.loc[i, 'node_name']
    i_idx = disease_nodes.loc[i, 'node_idx']
    # Exclude chromosome diseases. Terms like "Trisomy 21", "Chromosome 5 deletion" should not merge
    # — each is a distinct disease. They go into no (blacklist).
    for w in ['monosomy','disomy', 'trisomy', 'trisomy/tetrasomy', 'chromosome']: 
        if w in i_name: 
            no.add(i_idx)

for i in range(disease_nodes.shape[0]):
    i_idx = disease_nodes.loc[i, 'node_idx']
    if i_idx in seen:
        continue
    if i_idx in no:
        continue
    i_name = disease_nodes.loc[i, 'node_name']
    i_split = i_name.split(' ')
    end = i_split[-1]
    if has_subtype_suffix(end):
        main_text = ' '.join(i_split[:-1])
        matches = [i_name]
        matches_idx = [i_idx]
        match_found = False
        for j in range(disease_nodes.shape[0]):
            j_idx = disease_nodes.loc[j, 'node_idx']
            j_name = disease_nodes.loc[j, 'node_name']
            if rule_names_match(j_name, main_text):
                matches.append(j_name)
                matches_idx.append(j_idx)
                match_found = True
        if match_found:
            matches_idx = list(set(matches_idx))
            matches = list(set(matches))
            if len(matches) <= 1: continue 
            if main_text.endswith('type'): 
                main_text = main_text[:-4]
            if main_text.endswith(','): 
                main_text = main_text[:-1]
            if main_text.endswith(' '): 
                main_text = main_text[:-1]
            # print(main_text)
            # for x in sorted(matches): 
            #     print('-  ',x)
            for x in matches_idx: 
                seen.add(x)
                idx2group[x] = main_text
            groups.append((main_text, matches_idx))

In [4]:
disease_nodes["group_name"] = disease_nodes["node_name"]
for row in disease_nodes.itertuples():
    if row.node_idx in idx2group:
        disease_nodes.loc[row.Index, "group_name"] = idx2group[row.node_idx]

disease_group_1 = (
    disease_nodes[["group_name"]]
    .drop_duplicates()
    .reset_index()
    .rename(columns={"index": "group_idx"})
)
disease_nodes = disease_nodes.merge(disease_group_1, on="group_name", how="left")


In [5]:
len(disease_nodes), disease_nodes.group_idx.nunique(), len(disease_group_1), len(idx2group), len(seen), len(no)

(26386, 21245, 21245, 5921, 5921, 377)

## 3. BERT embeddings & cosine similarity

In [6]:
def compute_bert_embeddings(texts, batch_size=32, model_name="emilyalsentzer/Bio_ClinicalBERT"):
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device).eval()
    input_tokens = tokenizer(texts, padding=True, return_tensors="pt", truncation=True, max_length=512)

    tmp_dir = Path("tmp_bert_embeds")
    if tmp_dir.is_dir():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir()

    chunks = []
    for start in tqdm(range(0, len(texts), batch_size), desc="BERT embed"):
        end = min(start + batch_size, len(texts))
        input_ids = input_tokens["input_ids"][start:end].to(device)
        attention_mask = input_tokens["attention_mask"][start:end].to(device)
        with torch.no_grad():
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            chunks.append(torch.mean(outputs[0], dim=1).cpu().numpy())
    embeds = np.concatenate(chunks)
    shutil.rmtree(tmp_dir)
    return embeds


input_text = list(disease_group_1["group_name"].values)
need_recompute = RECOMPUTE_EMBEDDINGS or not EMBED_PATH.is_file()
if not need_recompute:
    embeds = np.load(EMBED_PATH)
    if embeds.shape[0] != disease_group_1.shape[0]:
        print(
            f"Stale BERT cache ({embeds.shape[0]} rows vs {len(disease_group_1)} disease groups) "
            f"→ recomputing {EMBED_PATH.name}"
        )
        need_recompute = True
if need_recompute:
    embeds = compute_bert_embeddings(input_text)
    np.save(EMBED_PATH, embeds)

assert embeds.shape[0] == disease_group_1.shape[0], (
    f"Embedding rows ({embeds.shape[0]}) != disease_group_1 ({len(disease_group_1)})"
)
cos_sim = cosine_similarity(embeds, embeds)

# Reset BERT clustering state (fresh pass on disease_group_1 / group_idx)
bert_seen = set()
bert_groups = []
bert_idx2group = {}
bert_no = set()

auto_block_substrings = [
    "cardiomyopathy", "syndrome", "combined", "complement", "deficiency",
    "factor", "immunodeficiency", "monosomy", "disomy", "trisomy",
    "trisomy/tetrasomy", "chromosome", "neuroendocrine tumor",
    "neuroendocrine neoplasm", "cancer", "tumor", "neoplasm", "carcinoma",
    "lymphoma", "lipoma",
]
auto_block_suffixes = ["CDG"]
auto_block_prefixes = [
    "neurodevelopmental disorder", "glycogen storage disease",
    "congenital disorder of glycosylation", "qualitative or quantitative defects",
]

for i in range(disease_group_1.shape[0]):
    i_name = disease_group_1.loc[i, "group_name"]
    i_idx = disease_group_1.loc[i, "group_idx"]
    if any(w in i_name for w in auto_block_substrings):
        bert_no.add(i_idx)
        continue
    if any(i_name.endswith(w) for w in auto_block_suffixes):
        bert_no.add(i_idx)
        continue
    if any(i_name.startswith(w) for w in auto_block_prefixes):
        bert_no.add(i_idx)
        continue

cutoff = BERT_CUTOFF
seen = bert_seen
groups = bert_groups
idx2group = bert_idx2group
no = bert_no


## 4. Legacy PrimeKG lookup & BERT clustering

In [7]:
prime_kg = pd.read_csv(PRIMEKG_PATH, low_memory=False)
prime_kg[(prime_kg.x_source=="MONDO_grouped")].head(2)

,relation,display_relation,x_index,x_id,x_type,x_name,x_source,y_index,y_id,y_type,y_name,y_source
3083868,disease_phenotype_negative,phenotype absent,27158,13924_12592_14672_13460_12591_12536_30861_8146...,disease,osteogenesis imperfecta,MONDO_grouped,22304,365,effect/phenotype,Hearing impairment,HPO
3083869,disease_phenotype_negative,phenotype absent,27158,13924_12592_14672_13460_12591_12536_30861_8146...,disease,osteogenesis imperfecta,MONDO_grouped,22675,1249,effect/phenotype,Intellectual disability,HPO


In [8]:
def build_mondo_to_legacy_grouped_name(prime_kg_df):
    mondo_to_grouped = {}
    legacy_grouped_canonical_names = set()
    member_set_to_legacy_gid = {}
    legacy_name_to_gid_sets = {}
    for side in ("x", "y"):
        tcol, scol, icol, ncol = f"{side}_type", f"{side}_source", f"{side}_id", f"{side}_name"
        mask = (prime_kg_df[tcol] == "disease") & (prime_kg_df[scol] == "MONDO_grouped")
        for gid, gname in zip(
            prime_kg_df.loc[mask, icol].astype(str),
            prime_kg_df.loc[mask, ncol].astype(str),
        ):
            parts = [p for p in gid.split("_") if p]
            member_fs = frozenset(parts)
            legacy_grouped_canonical_names.add(gname)
            for part in parts:
                mondo_to_grouped[part] = gname
            member_set_to_legacy_gid[member_fs] = gid
            legacy_name_to_gid_sets.setdefault(gname, {})[gid] = member_fs
    return mondo_to_grouped, legacy_grouped_canonical_names, member_set_to_legacy_gid, legacy_name_to_gid_sets


(
    mondo_to_legacy_group,
    legacy_grouped_names,
    legacy_member_set_to_gid,
    legacy_name_to_gid_sets,
) = build_mondo_to_legacy_grouped_name(prime_kg)


def resolve_group_composite_id(node_ids, member_set_to_legacy_gid):
    node_ids = [str(x) for x in node_ids if str(x)]
    member_fs = frozenset(node_ids)
    if member_fs in member_set_to_legacy_gid:
        return member_set_to_legacy_gid[member_fs], "legacy_exact_members"
    return "_".join(node_ids), "bert_join"


def resolve_cluster_group_name(member_idxs, disease_nodes_df, mondo_to_legacy):
    subset = disease_nodes_df[disease_nodes_df["group_idx"].isin(member_idxs)]
    node_ids = subset["node_id"].astype(str).unique()
    legacy_names = {mondo_to_legacy[nid] for nid in node_ids if nid in mondo_to_legacy}
    if len(legacy_names) == 1:
        return next(iter(legacy_names)), "found", ""
    if len(legacy_names) > 1:
        return NAME_DECIDED_LATER, "conflict", "|".join(sorted(legacy_names))
    return NAME_DECIDED_LATER, "missing", ""


In [9]:
def rebuild_idx2group_from_review(review_df):
    """Rebuild BERT yes-assignments from saved review file."""
    mapping = {}
    for _, row in review_df[review_df["decision_auto"] == "yes"].iterrows():
        name = row["proposed_group_name_auto"]
        for v in str(row["cluster_member_idx"]).split("|"):
            mapping[int(v)] = name
    return mapping


if not RECOMPUTE_BERT_REVIEW and REVIEW_PATH.is_file():
    review_new = pd.read_csv(REVIEW_PATH)
    idx2group = rebuild_idx2group_from_review(review_new)
    print(
        f"Loaded saved review: {len(review_new):,} clusters | "
        f"BERT yes-mapped group_idx: {len(idx2group):,}"
    )
else:
    assert cos_sim.shape[0] == disease_group_1.shape[0]
    confirm_candidates = 0
    yes_count = 0
    pending_count = 0
    review_rows = []

    for i in range(disease_group_1.shape[0]):
        i_name = disease_group_1.loc[i, "group_name"]
        i_idx = disease_group_1.loc[i, "group_idx"]
        if i_idx in no or i_idx in seen:
            continue

        x = disease_group_1[cos_sim[i] > cutoff]
        if x.shape[0] <= 1:
            continue

        confirm_candidates += 1
        member_names = list(x["group_name"].values.reshape(-1))
        member_idxs = list(x["group_idx"].values.reshape(-1))
        main_text, legacy_status, legacy_conflict_names = resolve_cluster_group_name(
            member_idxs, disease_nodes, mondo_to_legacy_group
        )
        decision = "yes" if main_text != NAME_DECIDED_LATER else "pending"
        for v in member_idxs:
            seen.add(v)
        if decision == "yes":
            for v in member_idxs:
                idx2group[v] = main_text
            groups.append((main_text, member_idxs))
            yes_count += 1
        else:
            pending_count += 1
        review_rows.append({
            "anchor_group_idx": i_idx,
            "anchor_group_name": i_name,
            "decision_auto": decision,
            "proposed_group_name_auto": main_text,
            "legacy_lookup_status": legacy_status,
            "legacy_conflict_names": legacy_conflict_names,
            "cutoff": cutoff,
            "cluster_size": len(member_idxs),
            "cluster_member_idx": "|".join(map(str, member_idxs)),
            "cluster_member_name": " || ".join(member_names),
            "reviewer_final_group_name": "",
        })

    review_new = pd.DataFrame(review_rows)
    review_new.to_csv(REVIEW_PATH, index=False)
    print(
        f"BERT review saved: {len(review_new):,} clusters | "
        f"yes={yes_count:,} pending={pending_count:,} | {REVIEW_PATH}"
    )


Loaded saved review: 1,858 clusters | BERT yes-mapped group_idx: 1,895


## 5. Expert review — dup-name fixes + Claude (pre-computed)

Flow:
1. `review_new` (BERT) — base
2. Override anchors from `20260616_dup_name_group_fixes.csv` (replaces Hoai review)
3. Apply Claude round1 (merge) then round2 (`no`)
4. Manual anchor tweaks in `MANUAL_OVERRIDES` if needed

Section 6 also applies **node-level** fixes from the same CSV.


In [10]:
HOAI_COLS = [
    "anchor_group_idx", "anchor_group_name", "decision_auto",
    "proposed_group_name_auto", "cutoff", "cluster_size", "Expert review",
    "cluster_member_idx", "cluster_member_name", "reviewer_final_group_name",
]

MANUAL_OVERRIDES = {
    "supratentorial ependymoma, ZFTA fusion–positive": "Supratentorial Ependymoma",
}


def merge_hoai_into_review(review_new_df, hoai_path):
    hoai = pd.read_csv(hoai_path)[HOAI_COLS].copy()
    hoai["Expert review"] = hoai["Expert review"].str.strip().str.lower()
    hoai = hoai.dropna(subset=["Expert review"])
    hoai = hoai.drop_duplicates(subset=["anchor_group_name"], keep="first")

    merged = review_new_df.merge(hoai, on="anchor_group_name", how="left", suffixes=("_new", "_old"))
    dup_cols = [c.replace("_new", "") for c in merged.columns if c.endswith("_new")]
    for col in dup_cols:
        if f"{col}_old" in merged.columns:
            merged[col] = merged[f"{col}_old"].fillna(merged[f"{col}_new"])
            merged = merged.drop(columns=[f"{col}_new", f"{col}_old"])
    return merged, hoai


def load_claude_decisions(round1_path, round2_path):
    r1 = pd.read_csv(round1_path)[["anchor_group_name", "Expert review"]]
    merge_decisions = r1[r1["Expert review"].notna() & (r1["Expert review"] != "no")]
    merge_decisions = merge_decisions.drop_duplicates(subset=["anchor_group_name"], keep="first")

    r2 = pd.read_csv(round2_path)[["anchor_group_name", "Expert review"]]
    no_decisions = r2.drop_duplicates(subset=["anchor_group_name"], keep="first")

    return pd.concat([merge_decisions, no_decisions]).drop_duplicates(
        subset=["anchor_group_name"], keep="first"
    )


def apply_expert_decisions(review_base, claude_decisions, manual_overrides=None):
    out = review_base.merge(
        claude_decisions.rename(columns={"Expert review": "claude_review"}),
        on="anchor_group_name",
        how="left",
    )
    out["Expert review"] = out["claude_review"].fillna(out["Expert review"])
    out = out.drop(columns=["claude_review"])
    if manual_overrides:
        for anchor, resolved in manual_overrides.items():
            out.loc[out["anchor_group_name"] == anchor, "Expert review"] = resolved
    return out


def load_dup_name_fixes(fixes_path):
    fixes = pd.read_csv(fixes_path, dtype=str).fillna("")
    anchors = fixes[fixes["record_type"] == "anchor"].copy()
    nodes = fixes[fixes["record_type"] == "node"].copy()
    return anchors, nodes


def apply_anchor_fixes(review_df, anchor_fixes_df):
    out = review_df.copy()
    if "Expert review" not in out.columns:
        out["Expert review"] = pd.NA
    for _, row in anchor_fixes_df.iterrows():
        mask = out["anchor_group_name"] == row["anchor_group_name"]
        if mask.any():
            out.loc[mask, "Expert review"] = row["expert_review"]
    return out


anchor_fixes, node_group_fixes = load_dup_name_fixes(DUP_NAME_FIXES_CSV)

if USE_HOAI_REVIEW:
    review_base, hoai_dedup = merge_hoai_into_review(review_new, HOAI_REVIEW_CSV)
else:
    review_base = review_new.copy()
    review_base["Expert review"] = pd.NA
    hoai_dedup = pd.DataFrame(columns=["anchor_group_name"])

claude_decisions = load_claude_decisions(CLAUDE_ROUND1_CSV, CLAUDE_ROUND2_CSV)
review_final = apply_expert_decisions(review_base, claude_decisions, MANUAL_OVERRIDES)
review_final = apply_anchor_fixes(review_final, anchor_fixes)

anchors_new_vs_hoai = set(review_new["anchor_group_name"]) - set(hoai_dedup["anchor_group_name"])
pending = review_final["Expert review"].isna()
print(
    f"review_final: {len(review_final):,} clusters | "
    f"Hoai-covered: {review_final['Expert review'].notna().sum() - claude_decisions['anchor_group_name'].isin(review_base[review_base['Expert review'].notna()]['anchor_group_name']).sum():,} (approx) | "
    f"Claude-updated: {review_final['anchor_group_name'].isin(claude_decisions['anchor_group_name']).sum():,} | "
    f"still pending (NaN): {pending.sum():,} | "
    f"rejected (no): {(review_final['Expert review'] == 'no').sum():,}"
)
if pending.any():
    display(review_final.loc[pending, ["anchor_group_name", "decision_auto", "cluster_member_name"]].head(10))


review_final: 1,858 clusters | Hoai-covered: 93 (approx) | Claude-updated: 63 | still pending (NaN): 1,765 | rejected (no): 64


,anchor_group_name,decision_auto,cluster_member_name
0,attenuated familial adenomatous polyposis,pending,attenuated familial adenomatous polyposis || A...
1,Charcot-Marie-Tooth disease axonal,yes,Charcot-Marie-Tooth disease axonal || Charcot-...
2,"epidermolysis bullosa simplex 1C, localized",yes,"epidermolysis bullosa simplex 1C, localized ||..."
3,autoimmune limbic encephalitis,yes,autoimmune limbic encephalitis || autoimmune e...
4,cutaneous mastocytoma,yes,cutaneous mastocytoma || cutaneous mastocytosi...
5,pseudohypoparathyroidism,yes,pseudohypoparathyroidism || pseudopseudohypopa...
6,"myoclonic epilepsy, juvenile",pending,"myoclonic epilepsy, juvenile || myoclonic epil..."
7,benign familial infantile epilepsy,pending,benign familial infantile epilepsy || benign a...
8,"diabetes mellitus, permanent neonatal",pending,"diabetes mellitus, permanent neonatal || diabe..."
9,congenital muscular dystrophy,pending,congenital muscular dystrophy || congenital my...


## 6. Apply `group_name_2` from expert review

In [11]:
# Expert review: yes → proposed_group_name_auto; custom name → use as-is; no → do not merge (keep separate MONDO)
df_map = review_final[review_final["Expert review"].notna() & (review_final["Expert review"] != "no")].copy()
df_map["resolved_name"] = df_map.apply(
    lambda r: r["proposed_group_name_auto"] if r["Expert review"] == "yes" else r["Expert review"],
    axis=1,
)
expert_map = df_map.set_index("anchor_group_name")["resolved_name"].to_dict()

disease_group_1["group_name_2"] = ""
for row in disease_group_1.itertuples():
    if row.group_name in expert_map:
        disease_group_1.loc[row.Index, "group_name_2"] = expert_map[row.group_name]
    elif row.group_idx in idx2group:
        disease_group_1.loc[row.Index, "group_name_2"] = idx2group[row.group_idx]
    else:
        disease_group_1.loc[row.Index, "group_name_2"] = row.group_name

df_disease_group = disease_nodes.merge(disease_group_1, on="group_name", how="left")[
    ["node_id", "node_type", "node_name", "node_source", "group_name", "group_name_2"]
].rename(columns={"group_name": "group_name_auto", "group_name_2": "group_name_bert"})
df_disease_group["node_id"] = df_disease_group["node_id"].astype(str)

node_fix_map = (
    node_group_fixes[node_group_fixes["node_id"].astype(str).str.len() > 0]
    .drop_duplicates(subset=["node_id"], keep="last")
    .set_index("node_id")["canonical_group_name"]
    .to_dict()
)
for row in df_disease_group.itertuples():
    canon = node_fix_map.get(str(row.node_id))
    if canon:
        df_disease_group.loc[row.Index, "group_name_bert"] = canon

df_disease_group.to_csv(aux_path / f"{RUN_DATE}-kg_grouped_diseases.csv", index=False)


## 7. Apply `MONDO_grouped` to KG

`group_id_bert`: reuse legacy composite id when member set matches PrimeKG; else `'_'.join(node_id)`.


In [12]:
grouped_diseases = pd.read_csv(aux_path / f"{RUN_DATE}-kg_grouped_diseases.csv", dtype={"node_id": str})
group_col = "group_name_bert"
id_col = "group_id_bert"

multi_groups = grouped_diseases.groupby(group_col).filter(lambda g: len(g) > 1)[group_col].unique()
set_groups = set(multi_groups)

group_map = pd.DataFrame({group_col: list(set_groups)})
group_map[id_col] = pd.NA
group_map["group_id_source"] = pd.NA

subset = grouped_diseases[grouped_diseases[group_col].isin(set_groups)]
id_source_counts = {}
for g, data in subset.groupby(group_col):
    composite_id, id_source = resolve_group_composite_id(
        list(data["node_id"].values), legacy_member_set_to_gid
    )
    group_map.loc[group_map[group_col] == g, id_col] = composite_id
    group_map.loc[group_map[group_col] == g, "group_id_source"] = id_source
    id_source_counts[id_source] = id_source_counts.get(id_source, 0) + 1

grouped_diseases = grouped_diseases.merge(group_map, on=group_col, how="left")
grouped_diseases.to_csv(aux_path / f"{RUN_DATE}-kg_grouped_diseases_bert_map.csv", index=False)


In [13]:
node_to_group = {
    (str(row["node_id"]), str(row["node_name"])): (str(row[id_col]), str(row[group_col]))
    for _, row in grouped_diseases.iterrows()
    if pd.notna(row.get(id_col)) and str(row[group_col]) in set_groups
}

for side in ("x", "y"):
    mask = (kg[f"{side}_type"] == "disease") & (kg[f"{side}_source"] == "MONDO")
    keys = list(zip(kg.loc[mask, f"{side}_id"].astype(str), kg.loc[mask, f"{side}_name"].astype(str)))
    kg.loc[mask, f"{side}_id"] = [node_to_group.get(k, (k[0],))[0] for k in keys]
    kg.loc[mask, f"{side}_name"] = [node_to_group.get(k, (k[0], k[1]))[1] if k in node_to_group else k[1] for k in keys]
    kg.loc[mask, f"{side}_source"] = ["MONDO_grouped" if k in node_to_group else "MONDO" for k in keys]

remaining = (
    kg.loc[kg["x_source"] == "MONDO", "x_id"].isin([k[0] for k in node_to_group]).sum()
    + kg.loc[kg["y_source"] == "MONDO", "y_id"].isin([k[0] for k in node_to_group]).sum()
)
assert remaining == 0, f"MONDO nodes still ungrouped: {remaining}"

kg = kg.drop_duplicates()
kg_rev = kg.rename(columns={
    "x_id": "y_id", "x_type": "y_type", "x_name": "y_name", "x_source": "y_source",
    "y_id": "x_id", "y_type": "x_type", "y_name": "x_name", "y_source": "x_source",
})
kg = pd.concat([kg, kg_rev]).drop_duplicates().dropna()
kg = kg.query(
    "not ((x_id == y_id) and (x_type == y_type) and (x_source == y_source) and (x_name == y_name))"
)
kg.to_csv(aux_path / f"{RUN_DATE}-kg_grouped.csv", index=False)


In [14]:
nodes = pd.concat([
    kg[["x_id", "x_type", "x_name", "x_source"]].rename(
        columns={"x_id": "node_id", "x_type": "node_type", "x_name": "node_name", "x_source": "node_source"}
    ),
    kg[["y_id", "y_type", "y_name", "y_source"]].rename(
        columns={"y_id": "node_id", "y_type": "node_type", "y_name": "node_name", "y_source": "node_source"}
    ),
]).drop_duplicates().reset_index(drop=True).reset_index().rename(columns={"index": "node_index"})

kg = kg.merge(
    nodes.rename(columns={"node_index": "x_index", "node_id": "x_id", "node_type": "x_type", "node_name": "x_name", "node_source": "x_source"}),
    how="left",
).dropna()
kg = kg.merge(
    nodes.rename(columns={"node_index": "y_index", "node_id": "y_id", "node_type": "y_type", "node_name": "y_name", "y_source": "y_source"}),
    how="left",
).dropna()
kg = kg[[
    "relation", "display_relation", "x_index", "x_id", "x_type", "x_name", "x_source",
    "y_index", "y_id", "y_type", "y_name", "y_source",
]]
edges = kg[["relation", "display_relation", "x_index", "y_index"]].copy()

kg.to_csv(save_path / f"{RUN_DATE}-primekg_plus.csv", index=False)
nodes.to_csv(save_path / f"{RUN_DATE}-nodes.csv", index=False)
edges.to_csv(save_path / f"{RUN_DATE}-edges.csv", index=False)

print(f"Saved: {save_path / f'{RUN_DATE}-primekg_plus.csv'}")
print(f"Nodes: {len(nodes):,} | Edges: {len(edges):,} | MONDO_grouped names: {grouped_diseases[group_col].nunique():,}")


Saved: /Users/ljw303/YANG_DATA/PrimeKG/datasets/data/kg/20260529-primekg_plus.csv
Nodes: 129,317 | Edges: 7,683,206 | MONDO_grouped names: 19,884
